### Load required libraries

In [ ]:
import requests
import json
import time
import itertools
import random
from pathlib import Path
import pandas as pd

requests 2.32.3
pandas   2.2.2
Session ready, User-Agent: BDACA-II-course/1.0 (educational; contact@example.com)


### Set up MediaWiki API

In [ ]:
# Define the API endpoint and polite defaults
API_URL = "https://en.wikipedia.org/w/api.php" # MediaWiki JSON API base URL
USER_AGENT = "BDACA-II-course/1.0 (educational;adashiyj@gmail.com)"
REQUEST_DELAY = 0.5  # seconds between requests
TIMEOUT = 30

session = requests.Session() # session persists certain parameters across multiple requests (more efficient than making individual requests)
session.headers.update({"User-Agent": USER_AGENT}) # this tells Wikipedia who is making the requests

def api_get(params: dict) -> dict | None: # Returns either a dictionary or None
    """Make a safe request to the MediaWiki API with retries and rate limiting."""
    try:
        response = session.get(API_URL, params=params, timeout=TIMEOUT)
        response.raise_for_status() # checks if the HTTP request was successful 
        data = response.json()
    except requests.exceptions.RequestException as e: 
        print(f"Request error: {e}")
        return None
    except json.JSONDecodeError: 
        print("Response was not valid JSON")
        return None
    finally: # this block always executes, whether an exception occurred or not
        time.sleep(REQUEST_DELAY)

    return data

### Test if convokit works

In [1]:
!pip3 install convokit

In [4]:
import spacy
from convokit import Corpus, TextParser, PolitenessStrategies
from convokit.model import Utterance, Speaker

# load spacy
nlp = spacy.load("en_core_web_sm")

# initialize
ps = PolitenessStrategies()

# create a single test utterance
utt = Utterance(id="0", speaker=Speaker(id="test"), text="Could you please help me with this?")

# correct argument is spacy_nlp not spacy_parse
result = ps.transform_utterance(utt, spacy_nlp=nlp, markers=True)
print("Politeness strategies:", result.meta["politeness_strategies"])

Politeness strategies: {'feature_politeness_==Please==': 1, 'feature_politeness_==Please_start==': 0, 'feature_politeness_==HASHEDGE==': 0, 'feature_politeness_==Indirect_(btw)==': 0, 'feature_politeness_==Hedges==': 0, 'feature_politeness_==Factuality==': 0, 'feature_politeness_==Deference==': 0, 'feature_politeness_==Gratitude==': 0, 'feature_politeness_==Apologizing==': 0, 'feature_politeness_==1st_person_pl.==': 0, 'feature_politeness_==1st_person==': 0, 'feature_politeness_==1st_person_start==': 0, 'feature_politeness_==2nd_person==': 1, 'feature_politeness_==2nd_person_start==': 0, 'feature_politeness_==Indirect_(greeting)==': 0, 'feature_politeness_==Direct_question==': 0, 'feature_politeness_==Direct_start==': 0, 'feature_politeness_==HASPOSITIVE==': 0, 'feature_politeness_==HASNEGATIVE==': 0, 'feature_politeness_==SUBJUNCTIVE==': 1, 'feature_politeness_==INDICATIVE==': 0}
